In [1]:
import firedrake as fd

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified



# The Poisson Equation

**1. The Strong Form**

Our first example regards the Poisson problem

\begin{equation}
\begin{cases}
    \begin{aligned}
    -\Delta u &= f \; &&\text{in} \; \Omega \\
    u &= u_{0}, \; &&\text{on} \; \partial \Omega
    \end{aligned}
\end{cases}
\end{equation}

Here, $u=u(x)$ is the unknown function, $f=f(x)$ is a prescribed function, $\Delta = \nabla \cdot \nabla (\cdot)$ stands for the Laplace operator (also often written as $\nabla^2(\cdot)$), $\Omega$ is the spatial domain, and $\partial \Omega$ is the boundary of $\Omega$. This equation arises in numerous physical contexts, including heat conduction, electrostatics, diffusion of substances, twisting of elastic rods, inviscid fluid flow, and water waves. 

**2. The Variational (or Weak) Form**

Introduce the trial and test spaces

\begin{equation}
\begin{aligned}
    U &= \{u \in H^1(\Omega): u=u_0\; \text{on}\; \partial \Omega\} \\
    V &= \{v \in H^1(\Omega): v=0\; \text{on}\; \partial \Omega\}
\end{aligned}
\end{equation}

Multiplying Eq.(1) by $v\in V$, integrating over $\Omega$, and applying the Divergence Theorem gives

\begin{equation}
\begin{aligned}
    \int_{\Omega}\nabla u \cdot \nabla v\,dx - \int_{\partial \Omega}\dfrac{\partial u}{\partial n}\,v\,ds = \int_{\Omega}f\,vdx, \quad \forall v\in V
\end{aligned}
\end{equation}

where $\partial u/\partial n$ is the outward normal derivative of $u$ on $\partial \Omega$. Since $v=0$ on $\partial \Omega$, the boundary term vanishes and the weak form reduces to:

Find $u \in U$ such that  
\begin{equation}
\boxed{\int_{\Omega}\nabla u \cdot \nabla v\,dx = \int_{\Omega}f\,vdx, \quad \forall v\in V}
\end{equation}

**3. The Discrete Variational (Galerkin) Form**

Replacing $U$ and $V$ with finite-dimensional spaces $U_h \subset U$ and $V_h \subset V$ — fixed by the choice of finite element (e.g., piecewise-linear on a triangular mesh) — gives the discrete problem:

Find $u_h \in U_h$ such that
\begin{equation}
\boxed{\int_{\Omega}\nabla u_h \cdot \nabla v_h\,dx = \int_{\Omega}f\,v_hdx, \quad \forall v_h\in V_h}
\end{equation}

Writing this as $a(u_h,v_h) = L(v_h)$, with bilinear form $a(u,v) = \int_{\Omega}\nabla u \cdot \nabla v\,dx$ and linear form $L(v) = \int_{\Omega}f\,vdx$, is exactly what is assembled in the Firedrake code below.


In [2]:
# Create the domain
mesh = fd.UnitSquareMesh(32, 32)

# Function space
U = fd.FunctionSpace(mesh, "CG", 1)
V = fd.FunctionSpace(mesh, "CG", 1)

# Trial and test functions
u = fd.TrialFunction(U)
v = fd.TestFunction(V)

# Source term 
f = fd.Constant(-6.0)

# Boundary conditions
x = fd.SpatialCoordinate(mesh)
u_bc = 1 + x[0]**2 + 2 * x[1]**2

bc = fd.DirichletBC(U, u_bc, "on_boundary")

# Bilinear and linear forms
a = fd.inner(fd.grad(u), fd.grad(v)) * fd.dx
L = f * v * fd.dx

# Solve the variational problem
u_h = fd.Function(U, name="u_h")
fd.solve(a == L, u_h, bcs=[bc], solver_parameters={"ksp_type": "cg", "pc_type": "ilu"})

# Save the solution to a file
fd.VTKFile("solution.pvd").write(u_h)

**4. Remark: Controlling the Solution Process**

By default, Firedrake uses a sparse LU (direct) solver — robust for up to a few thousand unknowns (2D small 3D problems), but slow and memory-hungry beyond that. Iterative Krylov solvers scale much better for large problems. Since the Poisson equation yields a symmetric positive-definite matrix, the Conjugate Gradient (CG) method is the optimal choice, paired here with an Incomplete LU (ILU) preconditioner.

**5. Linear Variational Problem and Solver Objects**

`fd.solve(a == L, u_h, bcs=bc, solver_parameters=...)` is compact syntax for assembling a `LinearVariationalProblem` and solving it with a `LinearVariationalSolver`, as done explicitly below — useful when the solver object needs to be reused or configured beyond a single call.

In [ ]:
u_hh = fd.Function(U, name="u_hh")
problem = fd.LinearVariationalProblem(a, L, u_hh, bcs=[bc])
solver = fd.LinearVariationalSolver(problem, solver_parameters={"ksp_type": "cg", "pc_type": "ilu"})
solver.solve()

**6. A More Realistc Physical Problem: Deflection of a Membrane**

Consider the deflection $D(x,y)$ of an elastic circular membrane of radius $R$, subject to a localized perpendicular pressure load modeled as a Gaussian function. The governing PDE is

\begin{equation}
\begin{aligned}
    -T\Delta D = p(x,y) \quad \text{in} \; \Omega = \{(x,y) : x^2+y^2 \le R^2\},
\end{aligned}
\end{equation}

with

\begin{equation}
\begin{aligned}
    p(x,y) = \frac{A}{2\pi\sigma}\exp\left(-\frac{1}{2}\left(\frac{x-x_0}{\sigma}\right)^2 - \frac{1}{2}\left(\frac{y-y_0}{\sigma}\right)^2\right).
\end{aligned}
\end{equation}

Here, $T$ is the (constant) membrane tension, $p$ the external pressure load, $A$ the amplitude of the pressure, $(x_0,y_0)$ the location of the Gaussian peak, and $\sigma$ its "width". The membrane has no deflection at the boundary, giving $D=0$ there.

**Scaling to a Dimensionless Problem**

To ease verification, it helps to have an analytical solution. In the limit $\sigma \gg R$, $p \to A/(2\pi\sigma)$, and integrating the resulting axisymmetric equation in the radial coordinate $r \in [0,R]$ gives $D(r) = (R^2-r^2)A/(8\pi\sigma T)$. This yields a characteristic deflection scale $D_c = AR^2/(8\pi\sigma T)$, which we use to non-dimensionalize the problem.

Non-dimensionalizing takes two coordinated substitutions: a **length scale** $x = x_{\text{phys}}/R,\; y=y_{\text{phys}}/R$ — so that $x_{\text{phys}}=Rx$, the domain becomes the unit circle, and by the chain rule $\Delta_{\text{phys}} = R^{-2}\Delta$ — and a **deflection scale** $D(x_{\text{phys}},y_{\text{phys}}) = D_c\,w(x,y)$. Substituting both into $-T\Delta_{\text{phys}}D=p$ gives

\begin{equation}
\begin{aligned}
    -T\,R^{-2}D_c\,\Delta w = p(Rx,Ry) \quad\Longrightarrow\quad -\Delta w = \frac{R^2}{TD_c}\,p(Rx,Ry) = \frac{8\pi\sigma}{A}\,p(Rx,Ry),
\end{aligned}
\end{equation}

i.e. the equivalent dimensionless problem on the unit circle,

\begin{equation}
\begin{aligned}
    -\Delta w = f,
\end{aligned}
\end{equation}

with $w=0$ on the boundary and

\begin{equation}
\begin{aligned}
    f(x,y) = 4\exp\left(-\frac{1}{2}\left(\frac{Rx-x_0}{\sigma}\right)^2 - \frac{1}{2}\left(\frac{Ry-y_0}{\sigma}\right)^2\right),
\end{aligned}
\end{equation}

where, for notational convenience, the scaled coordinates keep the symbols $x,y$ — only these (not $x_0,y_0,\sigma$) are rescaled by $R$. The physical deflection is recovered via $D = D_c\,w = \dfrac{AR^2}{8\pi\sigma T}w$.


In [4]:
# Parameters for the problem
T = fd.Constant(10.0)           # tension
A = fd.Constant(1.0)            # pressure amplitude
R = fd.Constant(0.3)            # radius of the domain
sigma = fd.Constant(0.025)      # standard deviation of the Gaussian pressure

x0 = fd.Constant(0.0)           # load centered at the origin
y0 = fd.Constant(0.0)

# Mesh (disk domain)
mesh = fd.UnitDiskMesh(refinement_level=3)

# Function space 
W = fd.FunctionSpace(mesh, "CG", 1)
S = fd.FunctionSpace(mesh, "CG", 1)

# Trial and test functions
w = fd.TrialFunction(W)
s = fd.TestFunction(S)

# Source term  
x, y = fd.SpatialCoordinate(mesh)
g = 4 * fd.exp(-0.5 * ((R * x - x0)/sigma)**2 - 0.5 * ((R * y - y0)/sigma)**2)

# Boundary conditions
bc = fd.DirichletBC(W, 0.0, "on_boundary")

# Bilinear and linear forms 
a = fd.inner(fd.grad(w), fd.grad(s)) * fd.dx
L = g * s * fd.dx

# Solve the variational problem
w_h = fd.Function(W, name="w_h")
fd.solve(a == L, w_h, bcs=bc, solver_parameters={"ksp_type": "cg", "pc_type": "ilu"})

# Save the solution to a file
fd.VTKFile("solution_circle.pvd").write(w_h)